In [ ]:
import json
import logging
import numpy as np
import pandas as pd
from zenml import pipeline, step
from zenml.constants import DEFAULT_SERVICE_START_STOP_TIMEOUT
from zenml.integrations.mlflow.model_deployers.mlflow_model_deployer import MLFlowModelDeployer
from zenml.integrations.mlflow.services import MLFlowDeploymentService
from zenml.integrations.mlflow.steps import mlflow_model_deployer_step
from pydantic import BaseModel
from zenml.config import DockerSettings
# from zenml.constants import DEFAULT_SERVICE_START_STOP_TIMEOUT


@step(enable_cache=False)
def dynamic_import() :
    """Downloads the latest data from a mock API."""
    try:
        df = pd.read_csv("./data/insurance.csv")
        df = df.sample(n=10)
        if "region" in df.columns:
            df = df.drop("region", axis=1)

        if "sex" in df.columns:
            df['sex'] = df['sex'].replace({"male": 1, "female": 0})

        if "smoker" in df.columns:
            df['smoker'] = df['smoker'].replace({"yes": 1, "no": 0})

        # X = df.drop("charges", axis=1)
        # y = df["charges"]
        df =  df.drop("charges", axis=1)
        result = df.to_json(orient="split")
        return result
    except Exception as e:
        logging.error(e)
        raise e
    
class DeploymentTriggerConfig(BaseModel):
    """Parameters that are used to trigger the deployment"""
    min_accuracy: float = 0.1

@step
def deployment_trigger(accuracy: float, config: DeploymentTriggerConfig) -> bool:
    """Implements a simple model deployment trigger"""
    return accuracy > config.min_accuracy

@step(enable_cache=False)
def prediction_service_loader(
    pipeline_name: str, 
    pipeline_step_name: str, 
    running: bool = True,
    model_name: str = "model"  
) -> MLFlowDeploymentService: 
    """Get the prediction service started by the deployment pipeline."""
    model_deployer = MLFlowModelDeployer.get_active_model_deployer()
    
    existing_services = model_deployer.find_model_server(
        pipeline_name=pipeline_name,
        pipeline_step_name=pipeline_step_name,
        model_name=model_name, 
        running=running,
    )
    
    if not existing_services:
        raise RuntimeError(
            f"No MLflow prediction service found for pipeline '{pipeline_name}' "
            f"and step '{pipeline_step_name}'"
        )
    
    return existing_services[0]

@step
def predictor(
    service: MLFlowDeploymentService,
    data, 
) -> np.ndarray:
    """Run an inference request against a prediction service"""
    service.start(timeout=10)
    data_dict = json.loads(data)
    
    # معالجة البيانات
    data_dict.pop("columns", None)
    data_dict.pop("index", None)
    
    columns_for_df = [
        "age", "sex", "bmi",
        "children", "smoker"
    ]
    
    df = pd.DataFrame(data_dict["data"], columns=columns_for_df)
    df = df.astype({col: "float64" for col in columns_for_df})
    json_list = json.loads(json.dumps(list(df.T.to_dict().values())))
    data_array = np.array(json_list)
    
    prediction = service.predict(data_array)
    print("🔥 Prediction:", prediction)

    return prediction

docker_settings = DockerSettings(required_integrations=["mlflow", "sklearn"])

@pipeline(
    enable_cache=False,
    settings={
        "docker": docker_settings
    }
)
def continuous_deployment_pipeline(
    min_accuracy: float = 0.5,
    workers: int = 1,
    timeout: int = DEFAULT_SERVICE_START_STOP_TIMEOUT,
):
    """Continuous deployment pipeline with MLflow tracking"""
    from steps.laod_df import data_load
    from steps.train import train_step
    from steps.evaluate import evaluate_step
    
    # تحميل البيانات
    x_train, x_test, y_train, y_test = data_load()
    
    # تدريب المودل
    model = train_step(x_train, y_train)
    
    # تقييم المودل
    r2_score = evaluate_step(model, x_test, y_test)
    
    # قرار الـ deployment
    deployment_decision = deployment_trigger(
        accuracy=r2_score,
        config=DeploymentTriggerConfig(min_accuracy=min_accuracy)
    )
    
    # نشر المودل
    mlflow_model_deployer_step(
        model=model,
        deploy_decision=deployment_decision,
        workers=workers,
        timeout=timeout,
        )
#     )👉 ده بيعمل:

# ياخد الموديل من MLflow
# يشغله كـ API
# يعمله deployment

# 📌 النتيجة:

# بقى عندك Service شغال

@pipeline(enable_cache=False)
def inference_pipeline(pipeline_name: str, pipeline_step_name: str):
    batch_data = dynamic_import()
    model_deployment_service = prediction_service_loader(
        pipeline_name=pipeline_name,
        pipeline_step_name=pipeline_step_name,
        running=False,
    )
    predictor(service=model_deployment_service, data=batch_data)